# 1a

In [1]:
class Book:
    """Represent a book in the library catalog."""

    def __init__(
        self,
        title: str,
        author: str,
        pages: int,
        genres: set[str]
    ) -> None:
        self.title = title
        self.author = author
        self.pages = pages
        self.genres = {self.normalize_genre(genre) for genre in genres}

    def __str__(self) -> str:
        return f"{self.title} by {self.author} ({self.pages} pages)"

    def __repr__(self) -> str:
        return (
            f"Book({self.title!r}, {self.author!r}, "
            f"{self.pages!r}, {self.genres!r})"
        )

    def reading_time(self, pages_per_hour: float = 40) -> float:
        """Return the estimated reading time in hours."""
        return self.pages / pages_per_hour
        
    @staticmethod
    def normalize_genre(genre: str) -> str:
        """Remove surrounding whitespace and lowercase a genre."""
        return genre.strip().lower()

    @classmethod
    def from_line(cls, line: str) -> "Book":
        """Create a book from a semicolon-separated donation line."""
        title, author, pages, genres = line.split(";")
        return cls(
            title.strip(),
            author.strip(),
            int(pages),
            set(genres.split(","))
        )
    

In [2]:
book = Book("Dune", "Frank Herbert", 412, {"sci-fi"})

print(book)
print(repr(book))

Dune by Frank Herbert (412 pages)
Book('Dune', 'Frank Herbert', 412, {'sci-fi'})


In [3]:
book = Book("Dune", "Frank Herbert", 412, {"sci-fi"})

print(book.reading_time())
print(book.reading_time(pages_per_hour=50))

10.3
8.24


# 1b

In [4]:
books = [
    Book("Moon Garden", "Ana Lee", 200, {"sci-fi", "adventure"}),
    Book("The Last Planet", "Ana Lee", 320, {"sci-fi"}),
    Book("River Secrets", "Ben Chen", 240, {"mystery"}),
    Book("Hidden Trails", "Maya Park", 180, {"adventure"}),
    Book("Winter Letters", "Leo Kim", 280, {"romance"})
]

In [5]:
for number, book in enumerate(books, start=1):
    print(f"{number}. {book}")

1. Moon Garden by Ana Lee (200 pages)
2. The Last Planet by Ana Lee (320 pages)
3. River Secrets by Ben Chen (240 pages)
4. Hidden Trails by Maya Park (180 pages)
5. Winter Letters by Leo Kim (280 pages)


# 1c

In [6]:
print(Book.normalize_genre("  Science Fiction "))

messy_book = Book(
    "Test Book",
    "Test Author",
    100,
    {"  Sci-Fi ", "sci-fi", " CLASSIC "}
)
print(messy_book.genres)

science fiction
{'classic', 'sci-fi'}


# 2a

In [7]:
from dataclasses import dataclass

@dataclass
class Library:
    """Store books and search them by genre."""

    books: list[Book]

    def add(self, book: Book) -> None:
        self.books.append(book)

    def by_genre(self, genre: str) -> list[Book]:
        genre = Book.normalize_genre(genre)
        return [book for book in self.books if genre in book.genres]

    def find_books(
        self,
        genres: list[str],
        match_all: bool
    ) -> list[Book]:
        requested = {Book.normalize_genre(genre) for genre in genres}
        matches = []

        for book in self.books:
            if match_all:
                keep = True
                for genre in requested:
                    if genre not in book.genres:
                        keep = False
                        break
            else:
                keep = False
                for genre in requested:
                    if genre in book.genres:
                        keep = True
                        break

            if keep:
                matches.append(book)

        return matches

In [8]:
library = Library(books)
print(library)

for book in library.by_genre("sci-fi"):
    print(book.title)

Library(books=[Book('Moon Garden', 'Ana Lee', 200, {'adventure', 'sci-fi'}), Book('The Last Planet', 'Ana Lee', 320, {'sci-fi'}), Book('River Secrets', 'Ben Chen', 240, {'mystery'}), Book('Hidden Trails', 'Maya Park', 180, {'adventure'}), Book('Winter Letters', 'Leo Kim', 280, {'romance'})])
Moon Garden
The Last Planet


@dataclass automatically generates a repr method, and since Library has no custom str, printing it uses that representation, and the books inside its list are displayed usign each Book's repr.

# 2b

In [9]:
reading_times = {
    book.title: round(book.reading_time(), 1)
    for book in library.books
}

authors = {book.author for book in library.books}

print(reading_times)
print(authors)

{'Moon Garden': 5.0, 'The Last Planet': 8.0, 'River Secrets': 6.0, 'Hidden Trails': 4.5, 'Winter Letters': 7.0}
{'Leo Kim', 'Ben Chen', 'Ana Lee', 'Maya Park'}


There are fewer authors since one author may write more than one book in the library.

# 2c

In [10]:
library = Library(books)

genres = ["sci-fi", "adventure"]

print("All requested genres:")
for book in library.find_books(genres, match_all=True):
    print(book.title)

print("At least one requested genre:")
for book in library.find_books(genres, match_all=False):
    print(book.title)

All requested genres:
Moon Garden
At least one requested genre:
Moon Garden
The Last Planet
Hidden Trails


# 3a

In [11]:
class AudioBook(Book):
    """Represent a book with an audio recording."""

    def __init__(
        self,
        title: str,
        author: str,
        pages: int,
        genres: set[str],
        narrator: str,
        minutes: float
    ) -> None:
        super().__init__(title, author, pages, genres)
        self.narrator = narrator
        self.minutes = minutes

    def __str__(self) -> str:
        return (
            f"{super().__str__()}, narrated by {self.narrator}, "
            f"{self.minutes / 60:.1f} hours"
        )

    def reading_time(self, pages_per_hour: float = 40) -> float:
        """Return the listening time in hours."""
        return self.minutes / 60

In [12]:
audio = AudioBook(
    "Star Journey",
    "Nora Chen",
    300,
    {"sci-fi"},
    "Sam Rivera",
    600
)

print(audio)
print(audio.reading_time())

library.add(audio)

for book in library.by_genre("sci-fi"):
    print(book)

Star Journey by Nora Chen (300 pages), narrated by Sam Rivera, 10.0 hours
10.0
Moon Garden by Ana Lee (200 pages)
The Last Planet by Ana Lee (320 pages)
Star Journey by Nora Chen (300 pages), narrated by Sam Rivera, 10.0 hours


# 3b

In [13]:
donations = [
    "The Left Hand of Darkness;Ursula K. Le Guin;304; Sci-Fi ,Classic",
    "Pride and Prejudice;Jane Austen;432;Romance, classic",
    "Project Hail Mary;Andy Weir;476;sci-fi",
]

donated_books = [Book.from_line(line) for line in donations]

for book in donated_books:
    library.add(book)

for book in library.by_genre("classic"):
    print(book)

The Left Hand of Darkness by Ursula K. Le Guin (304 pages)
Pride and Prejudice by Jane Austen (432 pages)


# 3c

The output would be the same for Ana and Ben, since we only created a default list. Ana and Ben refer to the same list; changing one equals changing the other.

In [14]:
class ReadingList:
    def __init__(self, owner, books=[]):
        self.owner = owner
        self.books = books

    def add(self, book):
        self.books.append(book)


ana = ReadingList("Ana")
ben = ReadingList("Ben")

ana.add(Book("Dune", "Frank Herbert", 412, {"sci-fi"}))

print(len(ana.books), len(ben.books))

1 1


In [15]:
ana.books is ben.books

True

In [16]:
from dataclasses import dataclass, field

@dataclass
class ReadingList:
    """Store an owner's reading list."""

    owner: str
    books: list[Book] = field(default_factory=list)

    def add(self, book: Book) -> None:
        self.books.append(book)

    def total_pages(self) -> int:
        return sum(book.pages for book in self.books)

In [17]:
ana = ReadingList("Ana")
ben = ReadingList("Ben")

ana.add(Book("Dune", "Frank Herbert", 412, {"sci-fi"}))

print(len(ana.books), len(ben.books))
print(ana.books is ben.books)
print(ana.total_pages(), ben.total_pages())

1 0
False
412 0


The original output was 1 1 for both ana and ben. That's because they share one list books=[], appending a book to Ana also changes Ben's list.

The new class uses field(default_factory=list), which creates separate lists for each object, so we wouldn't change both lists at the same time.